# Neural E-commerce Data Validation

This notebook verifies the structural integrity and completeness of the exported datasets to ensure they meet the research requirements before downstream analysis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
df_users = pd.read_csv('../exports/demographic.csv')
df_events = pd.read_parquet('../exports/sequential.parquet')

print(f"Loaded {len(df_users)} demographic rows and {len(df_events)} sequential events.")

## 1. Demographic Schema Validation

In [ ]:
required_user_cols = [
    'customer_id', 'email', 'age_group', 'gender', 'city_tier',
    'account_age_days', 'lifetime_order_count', 'total_order_value',
    'avg_order_value', 'loyalty_tier', 'preferred_device', 'payment_method_saved'
]

missing_user_cols = [c for c in required_user_cols if c not in df_users.columns]
if missing_user_cols:
    print(f"❌ Missing demographic columns: {missing_user_cols}")
else:
    print("✅ All required demographic columns are present.")
    
# Check for nulls in critical demographic fields
critical_user_cols = ['customer_id', 'age_group', 'gender', 'city_tier', 'loyalty_tier']
null_counts = df_users[critical_user_cols].isnull().sum()
if null_counts.sum() > 0:
    print("\n❌ Found null values in critical demographic fields:")
    print(null_counts[null_counts > 0])
else:
    print("✅ No nulls in critical demographic fields.")

## 2. Sequential Behavioral Schema Validation

In [ ]:
required_event_cols = [
    'event_id', 'session_id', 'customer_id', 'event_timestamp', 'event_type',
    'page_url', 'time_on_page_sec'
]

missing_event_cols = [c for c in required_event_cols if c not in df_events.columns]
if missing_event_cols:
    print(f"❌ Missing sequential event columns: {missing_event_cols}")
else:
    print("✅ All core sequential event columns are present.")

null_events = df_events[['event_id', 'session_id', 'customer_id', 'event_type']].isnull().sum()
if null_events.sum() > 0:
    print("\n❌ Found null values in core event identifiers:")
    print(null_events[null_events > 0])
else:
    print("✅ No nulls in core event identifiers.")

## 3. Relational Integrity

In [ ]:
# Ensure all customer_ids in events exist in users table
valid_customers = set(df_users['customer_id'])
event_customers = set(df_events['customer_id'].dropna())

orphaned_customers = event_customers - valid_customers
if orphaned_customers:
    print(f"❌ Found {len(orphaned_customers)} customer_ids in events without a demographic record.")
else:
    print("✅ All events map to a valid demographic user.")